# Module 03: Advanced Data Handling, Integrity, & Preprocessing


## Module outline

* **Step 1: Missing Data Analysis & Simple Imputation**
  * Missingness Mechanisms: Missing Completely at Random (MCAR), Missing at Random (MAR), and Missing Not at Random (MNAR).
  * Traditional Cleaning: Listwise deletion (complete case analysis), Pairwise deletion, and arbitrary constant imputation.
  * **Scikit-Learn `SimpleImputer` Deep Dive:**
    * Numerical imputation strategies: `mean`, `median`.
    * Categorical imputation strategies: `most_frequent`, `constant`.
    * Missingness indicators: The `add_indicator` parameter for preserving missingness patterns as features.
* **Step 2: Categorical Encoding Techniques**
  * Nominal Encoding: One-Hot Encoding (handling the dummy variable trap with `drop='first'`), Binary Encoding, BaseN Encoding, and Hash Encoding (The Hashing Trick).
  * Ordinal Encoding: Mapping order-sensitive categories (`OrdinalEncoder`).
  * Target-Based Encoding: Target/Mean Encoding, Weight of Evidence (WoE) Encoding, and James-Stein Estimator (with smoothing parameters to avoid target leakage).
* **Step 3: Feature Scaling & Mathematical Transformations**
  * Scaling Foundations:
    * Min-Max Scaling (`MinMaxScaler`): Bound features to $[0, 1]$ or $[-1, 1]$.
    * Standardization (`StandardScaler`): Center to $\mu=0$ and scale to $\sigma=1$.
    * MaxAbsScaler: Scale by absolute maximum (ideal for sparse data).
    * Robust Scaling (`RobustScaler`): Scaling using Median and IQR to mitigate outlier influence.
  * Monotonic Transformations:
    * Logarithmic Transform (including zero with $\log(1+x)$; requires $x>-1$).
    * Reciprocal, Square Root, and Exponential transforms.
    * Power Transforms: Box-Cox (requires positive data) and Yeo-Johnson (handles negative/positive data).
    * Quantile Transformer: Mapping features to uniform or normal distributions.
* **Step 4: Modern Data Quality, Validation, & Drift**
  * **Schema Validation:** Enforcing strict data contracts using frameworks like `pandera` or `Great Expectations`.
  * **Data Drift:** Detecting shifts in independent variable distributions ($P(X)$) using Kolmogorov-Smirnov tests and Population Stability Index (PSI).
  * **Concept Drift:** Tracking changes in the relationship between features and targets ($P(Y|X)$).
  * Advanced Imputation: K-Nearest Neighbors Imputation (`KNNImputer`), Multivariate Imputation by Chained Equations (`IterativeImputer` / MICE), and Random Forest-based imputation (`MissForest`).
  * Outlier Engineering:
    * Mathematical Identification: Z-Score, Modified Z-Score (using Median Absolute Deviation), IQR-Fences.
    * Distance/Density Algorithms: Mahalanobis Distance, Minimum Covariance Determinant (MCD), Local Outlier Factor (LOF).
    * Model-Based Isolation: One-Class SVMs, Isolation Forests, and Elliptic Envelopes.
    * Outlier Remediation: Deletion, capping (Winsorization), and robust estimator utilization.
  * Handling Highly Imbalanced Data:
    * Oversampling: SMOTE, Borderline-SMOTE, ADASYN.
    * Undersampling: Tomek Links, Edited Nearest Neighbors (ENN), NearMiss (Versions 1, 2, and 3).
    * Hybrid pipelines: SMOTE + Tomek Links, SMOTE + ENN.


## Example Dataset


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

# Let's define our exact 10-employee dataset in Python to display it cleanly
data = {
    'Employee ID': ['Emp 1', 'Emp 2', 'Emp 3', 'Emp 4', 'Emp 5', 'Emp 6', 'Emp 7', 'Emp 8', 'Emp 9', 'Emp 10'],
    'Department': ['Sales', 'R&D', 'HR', 'R&D', 'Sales', 'R&D', 'HR', 'Sales', 'R&D', 'Sales'],
    'Performance Rating': ['Low', 'Medium', 'Medium', 'High', 'High', 'Low', 'High', 'Medium', 'High', 'Medium'],
    'Years at Company': [1, 3, 2, 5, 4, 1, 8, 3, 10, 3],
    'Monthly Salary': [3000, 4500, 4000, 6000, 5500, 3200, 7000, 4800, 12000, 5000],
    'Attrition (Left?)': ['Yes (1)', 'No (0)', 'No (0)', 'No (0)', 'No (0)', 'Yes (1)', 'No (0)', 'No (0)', 'No (0)', 'Yes (1)']
}

df = pd.DataFrame(data)
display(df)


,Employee ID,Department,Performance Rating,Years at Company,Monthly Salary,Attrition (Left?)
0,Emp 1,Sales,Low,1,3000,Yes (1)
1,Emp 2,R&D,Medium,3,4500,No (0)
2,Emp 3,HR,Medium,2,4000,No (0)
3,Emp 4,R&D,High,5,6000,No (0)
4,Emp 5,Sales,High,4,5500,No (0)
5,Emp 6,R&D,Low,1,3200,Yes (1)
6,Emp 7,HR,High,8,7000,No (0)
7,Emp 8,Sales,Medium,3,4800,No (0)
8,Emp 9,R&D,High,10,12000,No (0)
9,Emp 10,Sales,Medium,3,5000,Yes (1)


## Module 3: Advanced Data Handling, Integrity, & Preprocessing

Data in the real world is rarely clean. Before passing datasets into any machine learning model, we must handle missing values, encode non-numerical structures, scale our ranges, and preserve predictive integrity.

### Module 3: Step 1 - Missing Data Analysis

- **Definition:** Missing data means a value was not recorded or is unavailable. In Python datasets, it is usually shown as `NaN`.
- **Where we use it:** Salary, tenure, age, rating, survey answers, or any column where some records are incomplete.
- **Why we use it:** A missing value is not the same as zero. Before filling or deleting it, we first need to understand *why* it might be missing.

#### The Three Missingness Mechanisms

1. **Missing Completely at Random (MCAR)**
   * **Definition:** The value is missing because of a random accident. The missingness is not connected to any column and not connected to the missing value itself.
   * **Where we see it:** A file upload failed for a few random rows, or a scanner randomly skipped some salary entries.
   * **Why it matters:** MCAR is the safest missingness type. Deleting a few rows may be acceptable because the missing rows are not a special group.
   * **Dataset analogy:** Employee 2's salary is missing because the system randomly failed while saving that row. It did not happen because of department, salary amount, or performance.

2. **Missing at Random (MAR)**
   * **Definition:** The value is missing for a reason we can see in another recorded column. After using that known column, the missingness is not mainly about the hidden value itself.
   * **Where we see it:** Salary is missing more often for one department, location, age group, or job role.
   * **Why it matters:** We should use the related columns while filling the missing values. A single median for the whole dataset may be too simple.
   * **Dataset analogy:** HR employees are less likely to share salary. The missing salary is related to the recorded `Department` column, so department should be considered during imputation.

3. **Missing Not at Random (MNAR)**
   * **Definition:** The value is missing because of the value itself, even after checking the other columns.
   * **Where we see it:** Very low salaries or very high salaries are hidden because employees do not want to reveal them.
   * **Why it matters:** This is the hardest case. Normal imputation can be biased because the missing values are not similar to the visible values.
   * **Dataset analogy:** Employees with very low salary do not disclose salary. The missingness is caused by the hidden salary value itself.

**Simple memory rule:**

- **MCAR:** missing because of pure accident.
- **MAR:** missing because of another known column.
- **MNAR:** missing because of the missing value itself.


### Module 3: Step 1 - Handling Missing Values

After identifying why values may be missing, the next decision is how to handle them. There are two beginner-level choices here: deletion and simple imputation.

#### 1. Deletion

- **Definition:** Remove missing data instead of filling it.
- **Where we use it:** When only a small number of rows are missing and the missingness is not biased.
- **Why we use it:** It keeps only real observed values, but it can reduce data size and may remove an important group.

**Types of deletion:**

- **Listwise deletion:** Remove the whole row if any needed value is missing.
- **Pairwise deletion:** For each calculation, use only the rows where the required columns are present.

**Example:** If salary is missing for Employee 2, listwise deletion removes Employee 2 from the analysis. Pairwise deletion may still use Employee 2 for calculations that do not need salary.

#### 2. Simple Imputation using `SimpleImputer`

- **Definition:** Fill each missing value using one simple rule learned from the available values in that column.
- **Where we use it:** When a model requires a complete table, but we do not want to delete rows.
- **Why we use it:** It gives a clean input table for machine learning. It is simple and useful as a baseline, but it does not recover the true missing value.

**Common `SimpleImputer` strategies:**

- **`mean`:** Fill numerical missing values with the column average.
- **`median`:** Fill numerical missing values with the middle value. This is safer when salaries have outliers, because one very high salary does not pull the median too much.
- **`most_frequent`:** Fill categorical missing values with the most common category.
- **`constant`:** Fill missing values with a fixed value such as `"Missing"`.
- **`add_indicator=True`:** Add a new 1/0 column showing whether the value was originally missing.

**How median imputation works in this dataset:**

The code makes two salaries missing. `SimpleImputer(strategy='median')` does not guess each person's real salary. It looks at the salary values that are still available, finds the median salary, and uses that same median to fill every missing salary in that column.

For example, if the available training salaries have median `$4,900`, then:

- missing salary for Employee 2 becomes `$4,900`
- missing salary for Employee 9 becomes `$4,900`
- `Salary_Was_Missing?` becomes `1` for those employees, so the model still knows the salary was originally missing

**How train/test imputation should work:**

When we have train and test data, the imputer must learn the median from the **training data only**.

1. Fit on training data: learn the training salary median, for example `$4,900`.
2. Transform training data: fill training missing salaries with `$4,900`.
3. Transform test data: fill test missing salaries with the same `$4,900`.

We do not calculate a new median from the test column. The test data should behave like future unseen data, so it must use the rule learned from training data.


In [2]:
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# Let's create a copy of our 10-employee dataset with artificial missing values
df_missing = df.copy()
df_missing.loc[1, 'Monthly Salary'] = np.nan       # Emp 2 Salary is missing
df_missing.loc[8, 'Monthly Salary'] = np.nan       # Emp 9 Salary is missing
df_missing.loc[2, 'Performance Rating'] = np.nan   # Emp 3 Performance is missing

print("Dataset containing missing (NaN) values:")
display(df_missing)

# 1. Initialize Numerical Imputer using Median strategy + Missingness Indicators
num_imputer = SimpleImputer(strategy='median', add_indicator=True)

salary_imputed = num_imputer.fit_transform(df_missing[['Monthly Salary']])

salary_df = pd.DataFrame(
    salary_imputed,
    columns=['Imputed Salary', 'Salary_Was_Missing_Flag']
)

# 2. Initialize Categorical Imputer using Most Frequent (Mode) strategy
cat_imputer = SimpleImputer(strategy='most_frequent')
# Ensure we flatten the 2D transformed result to a 1D array
perf_imputed = cat_imputer.fit_transform(df_missing[['Performance Rating']]).flatten()

# Combine and display results
df_imputed_clean = df_missing.copy()
df_imputed_clean['Monthly Salary'] = salary_df['Imputed Salary']
df_imputed_clean['Salary_Was_Missing?'] = salary_df['Salary_Was_Missing_Flag']
df_imputed_clean['Performance Rating'] = perf_imputed

print("\nDataset after applying SimpleImputer and Missingness Indicators:")
display(df_imputed_clean)

Dataset containing missing (NaN) values:


,Employee ID,Department,Performance Rating,Years at Company,Monthly Salary,Attrition (Left?)
0,Emp 1,Sales,Low,1,3000.0,Yes (1)
1,Emp 2,R&D,Medium,3,NaN,No (0)
2,Emp 3,HR,NaN,2,4000.0,No (0)
3,Emp 4,R&D,High,5,6000.0,No (0)
4,Emp 5,Sales,High,4,5500.0,No (0)
5,Emp 6,R&D,Low,1,3200.0,Yes (1)
6,Emp 7,HR,High,8,7000.0,No (0)
7,Emp 8,Sales,Medium,3,4800.0,No (0)
8,Emp 9,R&D,High,10,NaN,No (0)
9,Emp 10,Sales,Medium,3,5000.0,Yes (1)



Dataset after applying SimpleImputer and Missingness Indicators:


,Employee ID,Department,Performance Rating,Years at Company,Monthly Salary,Attrition (Left?),Salary_Was_Missing?
0,Emp 1,Sales,Low,1,3000.0,Yes (1),0.0
1,Emp 2,R&D,Medium,3,4900.0,No (0),1.0
2,Emp 3,HR,High,2,4000.0,No (0),0.0
3,Emp 4,R&D,High,5,6000.0,No (0),0.0
4,Emp 5,Sales,High,4,5500.0,No (0),0.0
5,Emp 6,R&D,Low,1,3200.0,Yes (1),0.0
6,Emp 7,HR,High,8,7000.0,No (0),0.0
7,Emp 8,Sales,Medium,3,4800.0,No (0),0.0
8,Emp 9,R&D,High,10,4900.0,No (0),1.0
9,Emp 10,Sales,Medium,3,5000.0,Yes (1),0.0


### Module 3: Step 2 - Categorical Encoding Techniques

- **Definition:** Categorical encoding means converting text labels into numbers that a machine learning model can read.
- **Where we use it:** Columns like `Department`, `Gender`, `City`, `Performance Rating`, or any column with names/groups instead of pure numbers.
- **Why we use it:** Most ML models calculate using numbers. Encoding lets the model use category information without misunderstanding the original text.

There are two common category types:

- **Nominal categories:** No natural order. Example: `Sales`, `R&D`, `HR`. One department is not greater than another.
- **Ordinal categories:** Clear order. Example: `Low`, `Medium`, `High`.

So we should not encode every category column the same way. The encoding method depends on what the category means.


### Module 3: Step 2 - Nominal Encoding with One-Hot Encoding

- **Definition:** One-hot encoding creates separate 0/1 columns for each category.
- **Where we use it:** Unordered columns such as `Department`.
- **Why we use it:** It tells the model which group a row belongs to without creating a fake ranking like `HR = 1`, `R&D = 2`, `Sales = 3`.

For the `Department` column, one-hot encoding can create columns like this:

| Department | Department_HR | Department_R&D | Department_Sales |
|---|---:|---:|---:|
| HR | 1 | 0 | 0 |
| R&D | 0 | 1 | 0 |
| Sales | 0 | 0 | 1 |

Each row gets `1` only for the department it belongs to and `0` for the others.

#### Dummy Variable Trap

The dummy variable trap happens when one encoded column can be perfectly guessed from the other encoded columns.

Look at the three department columns above. For every employee, exactly one department column is `1`. That means:

```text
Department_HR + Department_R&D + Department_Sales = 1
```

So if we already know:

```text
Department_R&D = 0
Department_Sales = 0
```

then we do not need a separate `Department_HR` column. The employee must be from HR.

This is why, for some linear models, we drop one category column and keep the other columns. The dropped category becomes the **reference category**.

Example with HR dropped:

| Department | Department_R&D | Department_Sales | Meaning |
|---|---:|---:|---|
| HR | 0 | 0 | reference category |
| R&D | 1 | 0 | R&D employee |
| Sales | 0 | 1 | Sales employee |

Now the model can still identify all three departments:

- `R&D = 0` and `Sales = 0` means HR
- `R&D = 1` and `Sales = 0` means R&D
- `R&D = 0` and `Sales = 1` means Sales

**Where this matters:** Mainly for linear regression and logistic regression when an intercept is used. Tree-based models usually do not need this same dummy-trap correction.

**In the code below:** `OneHotEncoder(drop='first')` drops the first sorted category. In this dataset, that reference category is HR.

Reference: [scikit-learn OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).


In [3]:
from sklearn.preprocessing import OneHotEncoder

# 1. Initialize OneHotEncoder dropping the first category to avoid dummy variable trap
ohe = OneHotEncoder(drop='first', sparse_output=False)

# Fit and transform our Department column
dept_encoded = ohe.fit_transform(df[['Department']])

# Convert to a readable DataFrame
dept_encoded_df = pd.DataFrame(
    dept_encoded,
    columns=ohe.get_feature_names_out(['Department'])
)

print("Original Department Column:")
display(df[['Department']].head(5))

print(f"\nOne-Hot Encoded (reference category: {ohe.categories_[0][ohe.drop_idx_[0]]}):")
display(dept_encoded_df.head(5))

Original Department Column:


,Department
0,Sales
1,R&D
2,HR
3,R&D
4,Sales



One-Hot Encoded (reference category: HR):


,Department_R&D,Department_Sales
0,0.0,1.0
1,1.0,0.0
2,0.0,0.0
3,1.0,0.0
4,0.0,1.0


### Module 3: Step 2 — Ordinal Encoding (Order-Sensitive Mapping)

- **Definition:** Ordinal encoding assigns numbers in a meaningful category order.
- **Where we use it:** Low, Medium, and High performance ratings.
- **Why we use it:** To preserve rank when the model can use it appropriately.

For ordinal features (ordered labels like `Performance Rating` which ranges from `Low` to `High`), we can map them to ordered integers. This preserves order, not measured distances between the categories.

* **How it works:** We define a sequence representing the hierarchy: $$\text{Low} \rightarrow 0,\quad \text{Medium} \rightarrow 1,\quad \text{High} \rightarrow 2$$

**Meaning:** High gets 2 because it ranks above Medium, not because it represents twice the performance of Low or Medium. A linear model using these codes assumes equal numerical steps; consider whether that is suitable.


In [4]:
from sklearn.preprocessing import OrdinalEncoder

# Define the strict hierarchy
order_hierarchy = [['Low', 'Medium', 'High']]

# Initialize the Encoder with our hierarchy
ord_enc = OrdinalEncoder(categories=order_hierarchy)

# Transform the Performance Rating
df_ordinal = df.copy()
df_ordinal['Performance_Encoded'] = ord_enc.fit_transform(df[['Performance Rating']])

print("Original Performance vs Ordinal Encoded Integers:")
display(df_ordinal[['Performance Rating', 'Performance_Encoded']])

Original Performance vs Ordinal Encoded Integers:


,Performance Rating,Performance_Encoded
0,Low,0.0
1,Medium,1.0
2,Medium,1.0
3,High,2.0
4,High,2.0
5,Low,0.0
6,High,2.0
7,Medium,1.0
8,High,2.0
9,Medium,1.0


### Module 3: Step 3 — Feature Scaling & Mathematical Transformations

- **Definition:** Scaling changes numerical units or ranges; transformations change how values are spaced.
- **Where we use it:** Features such as salary in thousands and tenure in single digits.
- **Why we use it:** To keep feature scale from dominating distance calculations and to improve optimization in scale-sensitive models.

**Fit on training data:** These small examples show mechanics on one table. During model evaluation, learn scaling and transformation parameters from training data only.


### Module 3: Step 3 — Scaling Foundations: MinMax, Standard, & Robust Scalers

1. **Min-Max Scaling (`MinMaxScaler`)**
   * **Definition:** Rescale values relative to the observed minimum and maximum.
   * **Where we use it:** Putting training features on a common range such as 0 to 1.
   * **Why we use it:** To make ranges comparable while preserving order.
   * **Formula:** $$X_{\text{scaled}} = \frac{X - X_{\text{min}}}{X_{\text{max}} - X_{\text{min}}}$$
   * **Output Bounds:** Training values lie in $[0,1]$ by default. New values outside the training minimum/maximum can transform outside that interval unless clipping is enabled.
   * **Limitation:** Highly sensitive to outliers.

2. **Standardization (`StandardScaler`)**
   * **Definition:** Subtract the mean and divide by the standard deviation.
   * **Where we use it:** Preparing features for many linear, distance-based, or PCA workflows.
   * **Why we use it:** To express each value in units of standard deviation relative to its mean.
   * **Formula:** $$z = \frac{X - \mu}{\sigma}$$
   * **Output Bounds:** Nonconstant training features have mean 0 and population standard deviation 1; outputs are unbounded. This does not make a distribution normal.
   * **Limitation:** Still affected by extreme outliers when calculating mean and standard deviation.

3. **Robust Scaling (`RobustScaler`)**
   * **Definition:** Subtract the median and divide by the interquartile range.
   * **Where we use it:** Scaling features with extreme observations.
   * **Why we use it:** To base the center and scale on the middle of the data rather than the extremes.
   * **Formula:** $$X_{\text{scaled}} = \frac{X - \text{Median}}{\text{IQR}}$$
   * **Output Bounds:** Unbounded. It scales data using median and Interquartile Range, making it highly resistant to outliers.


**Salary examples:**

- Min-max: for $X=6000$, minimum 3000, maximum 12000: $(6000-3000)/(12000-3000)=0.333$.
- Standard: $\mu$ is the training mean and $\sigma$ its population standard deviation. Here $(6000-5500)/2455.20\approx0.204$. `StandardScaler` uses division by $n$ in variance, unlike the sample SD calculation in Module 1.
- Robust: using the library's interpolated quartiles, median = 4900 and IQR = $5875-4125=1750$. Thus $(6000-4900)/1750\approx0.629$. The manual median-of-halves example uses a different quartile convention.

Reference: [scikit-learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).


In [5]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# Prepare our scalers
min_max = MinMaxScaler()
standard = StandardScaler()
robust = RobustScaler()

# Create a copy of our dataset and scale salaries
df_scaled = df.copy()
df_scaled['Salary_MinMax'] = min_max.fit_transform(df[['Monthly Salary']])
df_scaled['Salary_Standard'] = standard.fit_transform(df[['Monthly Salary']])
df_scaled['Salary_Robust'] = robust.fit_transform(df[['Monthly Salary']])

print("Comparing Scaling Methods on Employee Monthly Salaries:")
display(df_scaled[['Employee ID', 'Monthly Salary', 'Salary_MinMax', 'Salary_Standard', 'Salary_Robust']])

Comparing Scaling Methods on Employee Monthly Salaries:


,Employee ID,Monthly Salary,Salary_MinMax,Salary_Standard,Salary_Robust
0,Emp 1,3000,0.000000,-1.018248,-1.085714
1,Emp 2,4500,0.166667,-0.407299,-0.228571
2,Emp 3,4000,0.111111,-0.610949,-0.514286
3,Emp 4,6000,0.333333,0.203650,0.628571
4,Emp 5,5500,0.277778,0.000000,0.342857
5,Emp 6,3200,0.022222,-0.936788,-0.971429
6,Emp 7,7000,0.444444,0.610949,1.200000
7,Emp 8,4800,0.200000,-0.285109,-0.057143
8,Emp 9,12000,1.000000,2.647444,4.057143
9,Emp 10,5000,0.222222,-0.203650,0.057143


### Module 3: Step 3 - Monotonic and Power Transformations

Before learning the names, understand the main idea.

A **transformation** means changing the values in a column using a rule.

Example:

```text
Original salary:     3000, 4000, 6000, 12000
Transformed salary:  new values after applying a formula
```

We do this when the original numbers are hard for a model to use directly. A transformation can make very large values less extreme, reduce skew, or make patterns easier to model.

#### What does monotonic mean?

- **Simple meaning:** The order of values stays the same.
- If Employee A has a lower salary than Employee B before transformation, A should still have a lower transformed value after transformation.

Example:

```text
3000 < 4000 < 6000 < 12000
```

After a log transform, the exact numbers change, but the order is still the same:

```text
log(3000) < log(4000) < log(6000) < log(12000)
```

So a monotonic transformation changes the spacing between values, but it does not mix up their order.

#### What is a logarithm?

A logarithm is a way to shrink large numbers more than small numbers.

Think of it like this:

```text
Small values change a little.
Very large values get compressed a lot.
```

For salaries, this is useful because one very high salary can dominate the scale.

Example:

| Salary | Log value, approximately |
|---:|---:|
| 3000 | 8.01 |
| 4000 | 8.29 |
| 6000 | 8.70 |
| 12000 | 9.39 |

In the original salaries, the gap between `3000` and `12000` is huge: `9000`.

After log transform, the gap becomes much smaller: about `1.38`.

That is the main reason we use log transform: it compresses very large values.

#### Log Transform

- **Definition:** Replace each positive value with its logarithm.
- **Where we use it:** Positive columns with a long right tail, such as salary, income, price, population, or sales amount.
- **Why we use it:** To reduce the effect of very large values and make the column less skewed.
- **Formula:** `new_value = log(original_value)`

Important: normal log cannot be used on zero or negative values.

```text
log(0) is not valid
log(negative number) is not valid for normal real-valued ML preprocessing
```

#### Log-Plus-One Transform

Sometimes a column has zero values, such as purchase count, complaint count, or number of visits.

For that case, we use:

```text
new_value = log(1 + original_value)
```

This works for zero because:

```text
log(1 + 0) = log(1) = 0
```

In Python, we usually write this as `np.log1p(X)` because it is accurate and clear.

#### Square Root Transform

- **Definition:** Replace each value with its square root.
- **Where we use it:** Count-like columns where values are nonnegative.
- **Why we use it:** It reduces large values, but more gently than log transform.

Example:

```text
sqrt(100) = 10
sqrt(10000) = 100
```

The large value becomes smaller, but not as aggressively as log.

#### Reciprocal Transform

- **Definition:** Replace each value with `1 / value`.
- **Where we use it:** Positive values where smaller original values should become larger transformed values.
- **Why we use it:** It can strongly reduce the effect of large values, but it also reverses the order.

Example:

```text
1 / 2 = 0.5
1 / 10 = 0.1
```

Here, `2` is smaller than `10`, but `0.5` is larger than `0.1`. So reciprocal transform must be used carefully.

#### Power Transforms: Box-Cox and Yeo-Johnson

Power transforms are automatic transformation methods. Instead of us choosing only log or square root, the method tries to find a useful power transformation for the data.

- **Box-Cox:** Works only when all values are positive.
- **Yeo-Johnson:** Can work with zero and negative values too.

**Where we use them:** When a numerical column is skewed and we want a more model-friendly shape.

**Why we use them:** They can reduce skew without manually guessing the best formula.

Simple idea:

```text
The method tries different transformation strengths and picks one that fits the data better.
```

#### Quantile Transformer

- **Definition:** It changes the values so the column follows a chosen shape, usually uniform or normal.
- **Where we use it:** When we want to force a feature into a smoother distribution.
- **Why we use it:** It can help some models, but it changes the spacing strongly, so interpretation becomes harder.

#### When should we not transform?

Do not transform just because the method exists. Transform only when it helps the model or the analysis.

Be careful when:

- the original scale is important for explanation
- the column has negative values and you are thinking about log transform
- the transformed values become harder to explain to students or business users
- the model already handles skew and outliers well

**Simple memory rule:**

- **Log:** compress very large positive values.
- **Log-plus-one:** same as log, but allows zero.
- **Square root:** gentle compression for nonnegative values.
- **Reciprocal:** strong change, reverses order.
- **Box-Cox:** automatic power transform for positive values only.
- **Yeo-Johnson:** automatic power transform that can handle zero and negative values.
- **Quantile transformer:** forces the feature into a selected distribution shape.


In [6]:
# Calculate log transform on original salaries
df_log = df.copy()
df_log['Salary_Log'] = np.log(df_log['Monthly Salary'])

print("Log-Transformed salaries (converting multiplicative scales to additive scales):")
display(df_log[['Employee ID', 'Monthly Salary', 'Salary_Log']])

Log-Transformed salaries (converting multiplicative scales to additive scales):


,Employee ID,Monthly Salary,Salary_Log
0,Emp 1,3000,8.006368
1,Emp 2,4500,8.411833
2,Emp 3,4000,8.294050
3,Emp 4,6000,8.699515
4,Emp 5,5500,8.612503
5,Emp 6,3200,8.070906
6,Emp 7,7000,8.853665
7,Emp 8,4800,8.476371
8,Emp 9,12000,9.392662
9,Emp 10,5000,8.517193


### Module 3: Step 4 - Data Quality, Drift, Outliers, and Imbalanced Data

This step is about checking whether the data is safe to use before training or prediction.

A model can fail for four common reasons:

1. The table is broken or invalid.
2. New input data looks different from training data.
3. Some values are extreme or suspicious.
4. One target class is much smaller than the other.

So Step 4 should be learned in this order:

```text
Data quality checks -> drift -> outliers -> imbalanced data -> advanced cleaning methods
```

Do not start with advanced names like SMOTE or Tomek Links. First understand the problem they are trying to solve.


### Module 3: Step 4 - Schema Validation and Drift

#### 1. Schema Validation

- **Definition:** Schema validation means checking whether a table follows the expected rules.
- **Where we use it:** Before model training, before prediction, and whenever a new batch of data arrives.
- **Why we use it:** A model expects the same kind of data it was trained on. Wrong columns, wrong data types, or impossible values can break predictions.

Simple schema rules for our employee dataset:

| Column | Rule |
|---|---|
| `Monthly Salary` | must be a positive number |
| `Years at Company` | cannot be negative |
| `Department` | must be one of `Sales`, `R&D`, `HR` |
| `Performance Rating` | must be one of `Low`, `Medium`, `High` |
| `Attrition (Left?)` | must be `Yes (1)` or `No (0)` |

Example problem:

```text
Monthly Salary = -5000
```

That is not a real salary. Schema validation should catch it before the model uses it.

#### 2. Data Drift

- **Definition:** Data drift means the input feature distribution changed compared with the training data.
- **Where we use it:** Monitoring future prediction data after a model is deployed.
- **Why we use it:** A model trained on one type of data may perform badly when the world changes.

Example:

```text
Training salaries: 3000 to 12000
New salaries:      20000 to 50000
```

The model is now seeing salaries that are very different from the training period. That is data drift.

In notation, data drift is often written as a change in `P(X)`. Here, `X` means the input features, such as salary, tenure, and department.

#### 3. Concept Drift

- **Definition:** Concept drift means the relationship between inputs and the target changed.
- **Where we use it:** Monitoring whether an old prediction rule is still true.
- **Why we use it:** Even if the input columns look similar, the meaning of those inputs may change over time.

Example:

```text
Before: low salary employees were more likely to leave.
Later: high salary employees also start leaving because of poor work-life balance.
```

The salary column may look normal, but the relationship between salary and attrition changed. That is concept drift.

In notation, concept drift is often written as a change in `P(Y | X)`. Here, `Y` means the target, such as attrition.

#### Reading the KS Drift Example Below

The code below compares old salaries with new salaries using the KS test.

- **KS statistic:** how far apart the two salary distributions are.
- **p-value:** whether the difference looks statistically unusual under the test assumptions.

A small p-value is a warning sign. It does not automatically prove the model is bad; it says the new input data is different enough to investigate.


### Module 3: Step 4 - Outliers and Imbalanced Data

#### 1. Outliers

- **Definition:** An outlier is a value that is unusually far from the rest of the data.
- **Where we use it:** Salary, tenure, transaction amount, age, or any numerical column.
- **Why we use it:** Outliers can be real important cases, data-entry mistakes, or values that distort model training.

Example:

```text
Most salaries: 3000 to 7000
One salary:    12000
```

The `12000` salary is high compared with the others. It may be valid, but we should inspect it.

Common ways to find outliers:

- **IQR rule:** values far below Q1 or far above Q3.
- **Z-score:** values many standard deviations away from the mean.
- **Modified Z-score:** similar idea, but uses the median and MAD, so it is more robust.

Common ways to handle outliers:

- **Keep:** if the value is real and important.
- **Correct:** if it is a data-entry mistake.
- **Cap/Winsorize:** replace very extreme values with a boundary value.
- **Remove:** only when there is a clear reason.
- **Use robust methods:** use models or scalers that are less sensitive to extremes.

Advanced names like LOF, Isolation Forest, and Mahalanobis Distance are just more specialized ways to detect unusual points. Learn the simple outlier idea first.

#### 2. Imbalanced Data

- **Definition:** Imbalanced data means one target class has many more rows than another target class.
- **Where we use it:** Attrition, fraud, disease detection, spam detection, rare event prediction.
- **Why we use it:** Accuracy can become misleading when one class dominates.

Example:

```text
Stayed: 97 employees
Left:    3 employees
```

A model can predict `Stayed` for everyone and still get 97% accuracy, but it completely fails to find employees who may leave.

Better metrics for imbalanced data:

- **Recall:** out of actual positive cases, how many did we catch?
- **Precision:** out of predicted positive cases, how many were correct?
- **F1-score:** balances precision and recall.
- **Confusion matrix:** shows correct and wrong predictions for each class.

#### 3. Oversampling

- **Definition:** Add more minority-class examples to the training data.
- **Why we use it:** To give the model more chances to learn the smaller class.

**SMOTE, in simple words:** SMOTE creates new synthetic minority examples between existing minority examples.

Example idea:

```text
Minority point A ---- new synthetic point ---- Minority point B
```

Use SMOTE only on the training data, not on validation or test data.

#### 4. Undersampling

- **Definition:** Remove some majority-class examples from the training data.
- **Why we use it:** To reduce majority-class dominance.

**Tomek Links, in simple words:** A Tomek Link is a close pair where one row is from the majority class and the other is from the minority class. They are near each other and can confuse the decision boundary.

Example idea:

```text
Stayed employee  <very close>  Left employee
```

A common Tomek Links strategy removes the majority-class row from that confusing pair.

**ENN, in simple words:** Edited Nearest Neighbors removes rows whose nearby neighbors mostly belong to another class.

**NearMiss, in simple words:** NearMiss keeps majority-class examples that are close to minority-class examples, so the model focuses on the difficult boundary area.

#### Important Rule

Resampling must happen only inside the training data. Do not oversample or undersample the test set, because the test set should represent real future data.


In [7]:
from scipy.stats import ks_2samp

# Let's simulate new incoming salary data (Data Drift example)
baseline_salary = df['Monthly Salary']
new_drifted_salary = [12000, 15000, 11000, 9500, 13000, 14000, 8500, 11500, 16000, 10500]

# Perform a Kolmogorov-Smirnov test to detect drift
ks_stat, p_val = ks_2samp(baseline_salary, new_drifted_salary)

print(f"Baseline Salary Mean: ${baseline_salary.mean():,.2f}")
print(f"New Salary Mean: ${sum(new_drifted_salary)/len(new_drifted_salary):,.2f}")
print(f"KS Statistic: {ks_stat:.4f}, p-value: {p_val:.6f}")

if p_val < 0.05:
    print("Result: Significant Data Drift Detected! The new incoming dataset has a statistically different distribution.")
else:
    print("Result: No significant drift detected.")

Baseline Salary Mean: $5,500.00
New Salary Mean: $12,100.00
KS Statistic: 0.9000, p-value: 0.000217
Result: Significant Data Drift Detected! The new incoming dataset has a statistically different distribution.
